# SOOP Korean to Chinese SRT and hard-sub videos

This notebook uses Google Drive for model cache, outputs, and resume state. First pass creates `ko.srt` and `zh.srt`; second pass uses a CSV file to render high-resolution hard-sub clips.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If this notebook is not already inside the project directory, set REPO_URL and run this cell.
REPO_URL = ''  # Example: 'https://github.com/your-name/kr-sc-srt.git'
PROJECT_DIR = '/content/kr-sc-srt'

import os, pathlib
if REPO_URL and not pathlib.Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}
if pathlib.Path(PROJECT_DIR).exists():
    %cd {PROJECT_DIR}
else:
    print('Set REPO_URL or upload/open this notebook from the project directory.')

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q -U pip
!python -m pip install -q -e . yt-dlp

In [ ]:
from getpass import getpass
import os

ROOT = '/content/drive/MyDrive/kr-sc-srt'
MODEL_CACHE = f'{ROOT}/models'

# First run: paste the VOD URL. Rerun after failure/restart: leave empty and set RESUME_LAST = True.
VOD_URL = ''
RESUME_LAST = True

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI-compatible API key: ')

os.makedirs(ROOT, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

In [ ]:
cmd = ['kr-sc-srt', 'prepare', '--root', ROOT, '--model-cache-dir', MODEL_CACHE]
if VOD_URL:
    cmd.append(VOD_URL)
elif RESUME_LAST:
    cmd.append('--resume-last')
else:
    raise ValueError('Set VOD_URL or enable RESUME_LAST')

print(' '.join(cmd))
!{' '.join(cmd)}

Create or edit the segment CSV in the job output directory. Format:

```csv
name,start,end
part-01,00:01:30,00:05:00
part-02,00:05:00,00:08:30.500
```

By default, `render --resume-last` looks for `<job-name>.csv` inside the saved output directory.

In [ ]:
cmd = ['kr-sc-srt', 'render', '--root', ROOT, '--resume-last']
print(' '.join(cmd))
!{' '.join(cmd)}